In [5]:
import os
from dotenv import load_dotenv

load_dotenv()
DATA_ROOT = os.getenv("DATA_ROOT")
print("DATA_ROOT:", DATA_ROOT)

id_folder = os.path.join(DATA_ROOT, "id")
print("\nContents of id/ folder:")
for item in sorted(os.listdir(id_folder)):
    full_path = os.path.join(id_folder, item)
    if os.path.isdir(full_path):
        print(f"  [DIR] {item} -> {sorted(os.listdir(full_path))}")
    else:
        print(f"  [FILE] {item}")

DATA_ROOT: C:/Admin - Vaishali/Academics/VITV/Project_4_1/data/DRIAMS_A/DRIAMS-A

Contents of id/ folder:
  [DIR] 2015 -> ['2015_clean.csv', '2015_strat.csv']
  [DIR] 2016 -> ['._2016-01-12_IDRES_AB_not_summarised.csv', '2016_clean.csv', '2016_strat.csv']
  [DIR] 2017 -> ['2017_clean.csv', '2017_strat.csv']
  [DIR] 2018 -> ['2018_clean.csv', '2018_strat.csv']


In [6]:
import pandas as pd
import glob

# Load all _clean.csv files across years
csv_paths = glob.glob(os.path.join(id_folder, "*", "*_clean.csv"))
print("Found files:")
for p in csv_paths:
    print(" ", p)

dfs = []
for p in csv_paths:
    year = os.path.basename(os.path.dirname(p))
    df_year = pd.read_csv(p)
    df_year["year_folder"] = year  # tag with source year in case no date column exists
    dfs.append(df_year)

metadata = pd.concat(dfs, ignore_index=True)
print("\nCombined shape:", metadata.shape)
print("\nColumn names:")
print(metadata.columns.tolist())

Found files:
  C:/Admin - Vaishali/Academics/VITV/Project_4_1/data/DRIAMS_A/DRIAMS-A\id\2015\2015_clean.csv
  C:/Admin - Vaishali/Academics/VITV/Project_4_1/data/DRIAMS_A/DRIAMS-A\id\2016\2016_clean.csv
  C:/Admin - Vaishali/Academics/VITV/Project_4_1/data/DRIAMS_A/DRIAMS-A\id\2017\2017_clean.csv
  C:/Admin - Vaishali/Academics/VITV/Project_4_1/data/DRIAMS_A/DRIAMS-A\id\2018\2018_clean.csv


C:\Users\vaish\AppData\Local\Temp\ipykernel_28220\1934303986.py:13: DtypeWarning: Columns (0: Ticarcillin-Clavulan acid) have mixed types. Specify dtype option on import or set low_memory=False.
  df_year = pd.read_csv(p)



Combined shape: (111257, 93)

Column names:
['code', 'species', 'laboratory_species', 'Piperacillin-Tazobactam', 'Meropenem', 'Ciprofloxacin', 'Cefepime', 'Cotrimoxazole', 'Ceftazidime', 'Amikacin', 'Levofloxacin', 'Imipenem', 'Tobramycin', 'Ceftriaxone', 'Colistin', 'Clindamycin', 'Amoxicillin-Clavulanic acid', 'Amoxicillin', 'Vancomycin', 'Penicillin', 'Metronidazole', 'Erythromycin', 'Tetracycline', 'Fluconazole', 'Amphotericin B', 'Caspofungin', 'Micafungin', 'Anidulafungin', 'Itraconazole', '5-Fluorocytosine', 'Voriconazole', 'Posaconazole', 'Ampicillin-Amoxicillin', 'Ertapenem', 'Cefpodoxime', 'Norfloxacin', 'Fosfomycin-Trometamol', 'Nitrofurantoin', 'Linezolid', 'Teicoplanin', 'Tigecycline', 'Daptomycin', 'Gentamicin', 'Gentamicin_high_level', 'Aztreonam', 'Minocycline', 'Meropenem_without_meningitis', 'Meropenem_with_meningitis', 'Chloramphenicol', 'Quinolones', 'Clarithromycin', 'Cefuroxime', 'Oxacillin', 'Cefazolin', 'Rifampicin', 'Fusidic acid', 'Streptomycin', 'Rifampicin_

In [7]:
import sys
sys.path.append('..')  # notebook is in notebooks/, so '..' reaches the amr-prediction root

from src.data.labels import build_label_matrix

isolates, labels = build_label_matrix(metadata)
print("Isolates shape:", isolates.shape)
print("Labels shape:", labels.shape)

print("\nLabel sums (count of R per antibiotic):")
print(labels.sum())

Isolates shape: (4927, 93)
Labels shape: (4927, 5)

Label sums (count of R per antibiotic):
Ciprofloxacin                  1516
Cotrimoxazole                  1675
Ceftriaxone                    1110
Amoxicillin-Clavulanic acid    1324
Ampicillin-Amoxicillin         2974
dtype: int64


In [8]:
target_species = ["Escherichia coli", "Staphylococcus aureus", 
                   "Klebsiella pneumoniae", "Pseudomonas aeruginosa"]

print("Unique species sample (to confirm exact naming):")
print(metadata["species"].dropna().unique()[:20])

print("\nCounts for target species:")
species_counts = metadata["species"].value_counts()
for s in target_species:
    matches = species_counts.filter(like=s.split()[0])  # rough match in case of naming variants
    print(f"\n{s}:")
    print(matches)

Unique species sample (to confirm exact naming):
<StringArray>
[             'MIX!Streptococcus pneumoniae',
                'Staphylococcus epidermidis',
                     'Enterococcus faecalis',
                      'Enterococcus faecium',
                        'Klebsiella oxytoca',
                    'Pseudomonas aeruginosa',
 'Streptococcus salivarius_ssp_thermophilus',
                  'Propionibacterium avidum',
                   'Propionibacterium acnes',
            'Staphylococcus saccharolyticus',
                   'Lactobacillus johnsonii',
                      'Streptococcus oralis',
                       'Lactobacillus iners',
                 'Streptococcus intermedius',
                    'Lactobacillus jensenii',
                     'Gardnerella vaginalis',
                   'MIX!Lelliottia amnigena',
                     'Streptococcus equinus',
                     'Klebsiella pneumoniae',
                     'Lactobacillus gasseri']
Length: 20, dtype

In [9]:
ecoli_species_name = "Escherichia coli"  # confirmed exact match

ecoli_df = metadata[metadata["species"] == ecoli_species_name]
print("E. coli isolate count:", len(ecoli_df))

non_antibiotic_cols = ["code", "species", "laboratory_species", "year_folder", 
                        "Unnamed: 0.1", "Unnamed: 0"]
antibiotic_cols = [c for c in metadata.columns if c not in non_antibiotic_cols]

print(f"\nTotal antibiotic columns: {len(antibiotic_cols)}")
print("\nPer-antibiotic non-null count and R/S balance (E. coli only):")
for ab in antibiotic_cols:
    non_null = ecoli_df[ab].notna().sum()
    if non_null > 0:
        value_counts = ecoli_df[ab].value_counts()
        print(f"\n{ab}: {non_null} non-null")
        print(value_counts)

E. coli isolate count: 7320

Total antibiotic columns: 87

Per-antibiotic non-null count and R/S balance (E. coli only):

Piperacillin-Tazobactam: 5003 non-null
Piperacillin-Tazobactam
S                   4449
R                    306
-                    133
I                     44
R(1), S(1)            42
I(1), S(1)            28
R(1), I(1), S(1)       1
Name: count, dtype: int64

Meropenem: 5003 non-null
Meropenem
S             4925
-               69
I(1), S(1)       6
R                3
Name: count, dtype: int64

Ciprofloxacin: 5003 non-null
Ciprofloxacin
S             3445
R             1371
I               95
I(1), S(1)      38
R(1), S(1)      35
-               17
R(1), I(1)       2
Name: count, dtype: int64

Cefepime: 5003 non-null
Cefepime
S             4051
I              504
R              335
R(1), I(1)      57
I(1), S(1)      31
-               17
R(1), S(1)       8
Name: count, dtype: int64

Cotrimoxazole: 5003 non-null
Cotrimoxazole
S             3293
R             159

In [10]:
abx = ['Ciprofloxacin', 'Cotrimoxazole', 'Levofloxacin', 'Ceftriaxone',
       'Amoxicillin-Clavulanic acid', 'Ampicillin-Amoxicillin', 'Cefpodoxime']

# Per-year isolate counts for E. coli (Decision 1 check)
print("E. coli isolates per year:")
print(ecoli_df['year_folder'].value_counts().sort_index())

# treat '-' as missing
mask = (ecoli_df[abx] != '-').all(axis=1)
print(f"\nComplete cases across all 7: {mask.sum()}")

# and check without Cefpodoxime
mask6 = (ecoli_df[abx[:-1]] != '-').all(axis=1)
print(f"Complete cases across 6 (no Cefpodoxime): {mask6.sum()}")

# also: how many complete cases fall in 2018 (your test set)?
print("\n2018 slice (7-antibiotic complete cases):")
print(ecoli_df[mask].groupby('year_folder').size())

print("\n2018 slice (6-antibiotic complete cases):")
print(ecoli_df[mask6].groupby('year_folder').size())

E. coli isolates per year:
year_folder
2015     151
2016    2012
2017    3187
2018    1970
Name: count, dtype: int64

Complete cases across all 7: 4393
Complete cases across 6 (no Cefpodoxime): 7241

2018 slice (7-antibiotic complete cases):
year_folder
2015      79
2016     935
2017    2133
2018    1246
dtype: int64

2018 slice (6-antibiotic complete cases):
year_folder
2015     150
2016    1992
2017    3156
2018    1943
dtype: int64


In [11]:
abx = ['Ciprofloxacin', 'Cotrimoxazole', 'Levofloxacin', 'Ceftriaxone',
       'Amoxicillin-Clavulanic acid', 'Ampicillin-Amoxicillin']

MISSING_TOKEN = '-'

# complete-case filter (6-antibiotic set) — reuse mask6 logic but scoped to just these 6
mask6 = (ecoli_df[abx] != MISSING_TOKEN).all(axis=1)
complete_df = ecoli_df[mask6].copy()
print("Complete-case count:", len(complete_df))

# check for any values outside clean S/R/I before mapping
print("\nUnique raw values across the 6 antibiotics (complete-case subset):")
for ab in abx:
    print(f"{ab}: {complete_df[ab].unique()}")

Complete-case count: 7241

Unique raw values across the 6 antibiotics (complete-case subset):
Ciprofloxacin: <StringArray>
['S', nan, 'R', 'I', 'R(1), S(1)', 'I(1), S(1)', 'R(1), I(1)']
Length: 7, dtype: str
Cotrimoxazole: <StringArray>
['S', nan, 'R', 'R(1), S(1)']
Length: 4, dtype: str
Levofloxacin: <StringArray>
['S', nan, 'R', 'I', 'R(1), S(1)', 'I(1), S(1)', 'R(1), I(1)']
Length: 7, dtype: str
Ceftriaxone: <StringArray>
['S', nan, 'R', 'I(1), S(1)', 'R(1), S(1)', 'I']
Length: 6, dtype: str
Amoxicillin-Clavulanic acid: <StringArray>
['S', nan, 'R', 'R(1), S(1)']
Length: 4, dtype: str
Ampicillin-Amoxicillin: <StringArray>
['S', nan, 'R', 'R(1), S(1)']
Length: 4, dtype: str


In [12]:
import numpy as np

def collapse_label(val):
    """Map raw DRIAMS antibiotic result to 0 (S) / 1 (R), collapsing mixed results."""
    if pd.isna(val) or val == MISSING_TOKEN:
        return np.nan
    if 'R' in val:       # any R present (incl. mixed like 'R(1), S(1)') -> Resistant
        return 1
    if 'I' in val:       # any I present but no R -> treat as Resistant (I merged into R per LABEL_MAP)
        return 1
    if 'S' in val:       # pure S
        return 0
    return np.nan  # catch-all safety net

# proper complete-case mask: exclude BOTH '-' and real NaN
mask6_fixed = ecoli_df[abx].notna().all(axis=1) & (ecoli_df[abx] != MISSING_TOKEN).all(axis=1)
complete_df = ecoli_df[mask6_fixed].copy()
print("Corrected complete-case count:", len(complete_df))

# apply label collapsing
labels = complete_df[abx].map(collapse_label)
print("\nAny NaNs remaining after mapping? (should be 0 for all)")
print(labels.isna().sum())

print("\nClass balance per antibiotic after mapping:")
print(labels.mean())  # proportion R

Corrected complete-case count: 4924

Any NaNs remaining after mapping? (should be 0 for all)
Ciprofloxacin                  0
Cotrimoxazole                  0
Levofloxacin                   0
Ceftriaxone                    0
Amoxicillin-Clavulanic acid    0
Ampicillin-Amoxicillin         0
dtype: int64

Class balance per antibiotic after mapping:
Ciprofloxacin                  0.307880
Cotrimoxazole                  0.339968
Levofloxacin                   0.307880
Ceftriaxone                    0.225426
Amoxicillin-Clavulanic acid    0.268684
Ampicillin-Amoxicillin         0.603371
dtype: float64


In [13]:
print("Label correlation matrix:\n")
print(labels.corr().round(3))

Label correlation matrix:

                             Ciprofloxacin  Cotrimoxazole  Levofloxacin  \
Ciprofloxacin                        1.000          0.294         1.000   
Cotrimoxazole                        0.294          1.000         0.294   
Levofloxacin                         1.000          0.294         1.000   
Ceftriaxone                          0.551          0.302         0.551   
Amoxicillin-Clavulanic acid          0.298          0.238         0.298   
Ampicillin-Amoxicillin               0.426          0.468         0.426   

                             Ceftriaxone  Amoxicillin-Clavulanic acid  \
Ciprofloxacin                      0.551                        0.298   
Cotrimoxazole                      0.302                        0.238   
Levofloxacin                       0.551                        0.298   
Ceftriaxone                        1.000                        0.331   
Amoxicillin-Clavulanic acid        0.331                        1.000   
Ampicilli

In [14]:
(labels['Ciprofloxacin'] == labels['Levofloxacin']).all()

np.True_

In [15]:
abx = ['Ciprofloxacin', 'Cotrimoxazole', 'Levofloxacin', 'Ceftriaxone',
       'Amoxicillin-Clavulanic acid', 'Ampicillin-Amoxicillin']

print("Cipro == Levo exactly?", (labels['Ciprofloxacin'] == labels['Levofloxacin']).all())

print("\nChecking all other pairs for duplicates:")
for i in range(len(abx)):
    for j in range(i+1, len(abx)):
        a, b = abx[i], abx[j]
        if (labels[a] == labels[b]).all():
            print(f"IDENTICAL: {a} / {b}")

Cipro == Levo exactly? True

Checking all other pairs for duplicates:
IDENTICAL: Ciprofloxacin / Levofloxacin


In [16]:
print("Per-year breakdown of corrected complete cases (6-antibiotic set):")
print(complete_df['year_folder'].value_counts().sort_index())

Per-year breakdown of corrected complete cases (6-antibiotic set):
year_folder
2015      96
2016    1377
2017    2078
2018    1373
Name: count, dtype: int64


In [17]:
abx7 = abx + ['Cefpodoxime']
mask7_fixed = ecoli_df[abx7].notna().all(axis=1) & (ecoli_df[abx7] != '-').all(axis=1)
print("Corrected complete-case count with Cefpodoxime (7 antibiotics):", mask7_fixed.sum())
print("Corrected complete-case count without Cefpodoxime (6 antibiotics):", len(complete_df))

Corrected complete-case count with Cefpodoxime (7 antibiotics): 2076
Corrected complete-case count without Cefpodoxime (6 antibiotics): 4924


In [18]:
abx5 = ['Ciprofloxacin', 'Cotrimoxazole', 'Ceftriaxone', 
        'Amoxicillin-Clavulanic acid', 'Ampicillin-Amoxicillin']

mask5 = ecoli_df[abx5].notna().all(axis=1) & (ecoli_df[abx5] != '-').all(axis=1)
print("Complete cases with 5 antibiotics (no Levofloxacin):", mask5.sum())

Complete cases with 5 antibiotics (no Levofloxacin): 4927


In [19]:
from src.config import DATA_ROOT, ANTIBIOTICS

In [24]:
import importlib
from src.data import load, split
importlib.reload(load)
importlib.reload(split)

from src.data.load import load_dataset
from src.data.split import temporal_split, random_split

dataset_v2 = load_dataset()
print("n_samples:", dataset_v2.n_samples)
print("columns:", dataset_v2.y.columns.tolist())

n_samples: 4644
columns: ['Amoxicillin-Clavulanic acid', 'species', 'laboratory_species', 'Ciprofloxacin', 'Ceftriaxone', 'Ampicillin-Amoxicillin', 'code', 'Cotrimoxazole']


In [26]:
from src.data.load import load_dataset

# We still need metadata_df with year_folder for temporal_split's year lookup
# (load_dataset()'s DRIAMSDataset doesn't carry year info)
import glob, os
import pandas as pd
from src.config import ID_FOLDER

csv_paths = glob.glob(os.path.join(ID_FOLDER, "*", "*_clean.csv"))
dfs = []
for p in csv_paths:
    year = os.path.basename(os.path.dirname(p))
    df_year = pd.read_csv(p, low_memory=False)
    df_year["year_folder"] = year
    dfs.append(df_year)
isolates_v2 = pd.concat(dfs, ignore_index=True)

print("isolates_v2 shape:", isolates_v2.shape)

isolates_v2 shape: (111257, 93)


In [28]:
import importlib
from src.data import split
importlib.reload(split)
from src.data.split import temporal_split, random_split

# Temporal split
split_temporal = temporal_split(dataset_v2, isolates_v2)

# Random split
split_random = random_split(dataset_v2)

temporal_split: train=3331, test=1313
random_split: train=3483, test=1161


In [29]:
print(iso_train_t.shape, lab_train_t.shape)
print(iso_test_t.shape, lab_test_t.shape)

NameError: name 'iso_train_t' is not defined

In [ ]:
print(isolates_v2['code'].head(10).tolist())

['e80cbb84-b64c-4642-b3af-b03bac700a3a', '1eb604e9-f8b8-432b-b588-17d74da1ef8d', 'b78d685d-bfdb-4978-a823-34d3056ba9f1', '7495d24a-a101-48ec-85c6-ee0fa22f7118', '3bc11ab2-c5a3-44af-a789-435677293bda', 'cf0d428e-06da-401b-b81a-d96fb3356e45', '77165bc4-a202-4135-98f4-6b228ebf02b8', '0fbfe069-8466-4051-8bc0-23aec0d67c7e', '1a8ed8b4-fa5e-4c15-befc-e01041d67168', '97a2bd8d-f471-4a9a-baad-ec57dc4f7de3']


In [ ]:
import pkgutil
import maldi_learn
for mod in pkgutil.iter_modules(maldi_learn.__path__):
    print(mod.name)

data
driams
exceptions
filters
kernels
metrics
preprocessing
utilities
vectorization


In [ ]:
import maldi_learn.driams as driams
help(driams)

Help on module maldi_learn.driams in maldi_learn:

NAME
    maldi_learn.driams - Main module for the DRIAMS data set.

DESCRIPTION
    This is the main module for the DRIAMS data set. It contains general
    exploration classes and loaders.

CLASSES
    builtins.object
        DRIAMSDataset
        DRIAMSDatasetExplorer
    maldi_learn.preprocessing.generic.LabelEncoder(sklearn.base.BaseEstimator, sklearn.base.TransformerMixin)
        DRIAMSLabelEncoder
    
    class DRIAMSDataset(builtins.object)
     |  DRIAMSDataset(X, y)
     |  
     |  DRIAMS data set.
     |  
     |  Methods defined here:
     |  
     |  __init__(self, X, y)
     |      Create new DRIAMS data set.
     |      
     |      Parameters
     |      ----------
     |      X:
     |          List of `MaldiTofSpectra` objects.
     |      y:
     |          Metadata data frame (`pandas.DataFrame`). Columns with
     |          antimicrobial information are indicated by capitalized
     |          header.
     |  
 

In [ ]:
import pkgutil
for mod in pkgutil.iter_modules(driams.__path__):
    print(mod.name)

AttributeError: module 'maldi_learn.driams' has no attribute '__path__'

In [ ]:
print(dir(driams))

['AntibioticNotFoundException', 'AntibioticNotFoundWarning', 'DRIAMSDataset', 'DRIAMSDatasetExplorer', 'DRIAMSFilter', 'DRIAMSLabelEncoder', 'DRIAMS_ROOT', 'LabelEncoder', 'MaldiTofSpectrum', 'SpeciesNotFoundException', 'SpeciesNotFoundWarning', 'SpectraNotFoundException', 'SpectraNotFoundWarning', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', '_check_id_file', '_load_metadata', '_merge_years', '_metadata_columns', '_raise_or_warn', 'collections', 'dotenv', 'hashlib', 'itertools', 'load_driams_dataset', 'load_spectrum', 'np', 'os', 'pd', 'warnings']


In [ ]:
root_for_driams = DATA_ROOT.parent  # .../DRIAMS_A, without the inner DRIAMS-A
print(root_for_driams)

dataset = load_driams_dataset(
    root=str(root_for_driams),
    site='DRIAMS-A',
    years=['2015', '2016', '2017', '2018'],
    species='Escherichia coli',
    antibiotics=ANTIBIOTICS,
    handle_missing_resistance_measurements='remove_if_any_missing',
    spectra_type='binned_6000',
)
print("n_samples:", dataset.n_samples)

for ab in ANTIBIOTICS:
    print(f"{ab}: {dataset.class_ratio(ab)}")

C:\Admin - Vaishali\Academics\VITV\Project_4_1\data\DRIAMS_A
n_samples: 4644


TypeError: DRIAMSDataset.class_ratio() missing 1 required positional argument: 'antibiotic'

In [ ]:
import inspect
from maldi_learn.driams import DRIAMSDataset

# Get the property object itself, not the instance's evaluated result
prop = DRIAMSDataset.class_ratio
print(type(prop))
print(inspect.getsource(prop.fget))  # fget = the actual getter function inside the property

<class 'property'>
    @property
    def class_ratio(self, antibiotic):
        # extract copy of series
        ab_series = self.y[antibiotic].dropna()
        # return dict with label as key, and class fraction as value
        return ab_series.value_counts(normalize=True).to_dict()



In [ ]:
print("n_samples:", dataset.n_samples)

print("\nColumns available in dataset.y:")
print(dataset.y.columns.tolist())

print("\nClass ratio per antibiotic (computed directly):")
for ab in ANTIBIOTICS:
    ratio = dataset.y[ab].dropna().value_counts(normalize=True)
    print(f"\n{ab}:")
    print(ratio)

n_samples: 4644

Columns available in dataset.y:
['Ampicillin-Amoxicillin', 'Amoxicillin-Clavulanic acid', 'Ciprofloxacin', 'code', 'species', 'laboratory_species', 'Cotrimoxazole', 'Ceftriaxone']

Class ratio per antibiotic (computed directly):

Ciprofloxacin:
Ciprofloxacin
0    0.709518
1    0.290482
Name: proportion, dtype: float64

Cotrimoxazole:
Cotrimoxazole
0    0.674419
1    0.325581
Name: proportion, dtype: float64

Ceftriaxone:
Ceftriaxone
0    0.791128
1    0.208872
Name: proportion, dtype: float64

Amoxicillin-Clavulanic acid:
Amoxicillin-Clavulanic acid
0    0.753445
1    0.246555
Name: proportion, dtype: float64

Ampicillin-Amoxicillin:
Ampicillin-Amoxicillin
1    0.582257
0    0.417743
Name: proportion, dtype: float64


In [ ]:
# Check: does dataset.y's index/codes match a subset of our isolates_v2's codes?
dataset_codes = set(dataset.y['code'])
our_codes = set(isolates_v2['code'])

print("Codes in dataset.y but not in our isolates_v2:", len(dataset_codes - our_codes))
print("Codes in our isolates_v2 but not in dataset.y:", len(our_codes - dataset_codes))
print("Codes in both:", len(dataset_codes & our_codes))

Codes in dataset.y but not in our isolates_v2: 0
Codes in our isolates_v2 but not in dataset.y: 283
Codes in both: 4644


In [ ]:
# Need year info - dataset.y doesn't include year_folder, so join it back from isolates_v2
dataset_codes_df = dataset.y[['code']].copy()
year_lookup = isolates_v2[['code', 'year_folder']]

dataset_with_year = dataset_codes_df.merge(year_lookup, on='code', how='left')

print("Per-year breakdown of the validated 4,644 set:")
print(dataset_with_year['year_folder'].value_counts().sort_index())

# Confirm no unmatched codes (should be 0 nulls)
print("\nAny codes that failed to match a year?", dataset_with_year['year_folder'].isna().sum())

Per-year breakdown of the validated 4,644 set:
year_folder
2015      89
2016    1290
2017    1952
2018    1313
Name: count, dtype: int64

Any codes that failed to match a year? 0


In [ ]:
print(dataset.y.columns.tolist())

['Ampicillin-Amoxicillin', 'Amoxicillin-Clavulanic acid', 'Ciprofloxacin', 'code', 'species', 'laboratory_species', 'Cotrimoxazole', 'Ceftriaxone']


In [ ]:
import importlib
from src.data import load
importlib.reload(load)
from src.data.load import load_dataset

dataset_v2 = load_dataset()
print("n_samples:", dataset_v2.n_samples)
print("columns:", dataset_v2.y.columns.tolist())

ModuleNotFoundError: No module named 'src'

In [ ]:
y = dataset_v2.to_numpy('Ciprofloxacin')
print("y shape:", y.shape)

# X is accessed separately - dataset.X is a list of MaldiTofSpectrum objects
print("Number of spectra in dataset.X:", len(dataset_v2.X))
print("Type of one spectrum:", type(dataset_v2.X[0]))

# check train/test split indices work against y
print("\nTrain y shape:", y[split_temporal['train_idx']].shape)
print("Test y shape:", y[split_temporal['test_idx']].shape)

NameError: name 'dataset_v2' is not defined

In [ ]:
spectrum = dataset_v2.X[0]
print("Attributes/methods:", [a for a in dir(spectrum) if not a.startswith('_')])

NameError: name 'dataset_v2' is not defined

In [ ]:
spectrum = dataset_v2.X[0]

print("shape:", spectrum.shape)
print("dtype:", spectrum.dtype)
print("n_peaks:", spectrum.n_peaks)

print("\nintensities shape:", spectrum.intensities.shape)
print("intensities sample:", spectrum.intensities[:10])

print("\nmass_to_charge_ratios shape:", spectrum.mass_to_charge_ratios.shape)
print("mass_to_charge_ratios sample:", spectrum.mass_to_charge_ratios[:10])

NameError: name 'dataset_v2' is not defined